# Tabular — Deteksi Fraud Transaksi (feature kecepatan transaksi + split waktu)
**Kapan dipakai:** tabel transaksi kartu/dompet digital dengan kolom waktu, id kartu,
nominal, dan label fraud yang sangat jarang (<1%).

**Idenya:** fraud biasanya terlihat dari **pola kecepatan**: tiba-tiba banyak transaksi
dalam sejam, nominal jauh di atas kebiasaan, jeda sangat pendek.

**Langkah:**
1. Untuk tiap transaksi, hitung feature dari transaksi SEBELUMNYA milik kartu yang sama
   (jumlah dan total dalam 1 jam / 24 jam / 7 hari, jam sejak transaksi terakhir, rasio
   nominal vs rata-rata kartu).
2. Split berdasarkan WAKTU (bukan acak).
3. LGBMClassifier → PR-AUC → pilih threshold.

**Aturan emas:** feature hanya boleh memakai masa lalu (`closed="left"`, `shift(1)`). Kalau
tidak, skor validasi bagus palsu.

**Data demo:** 133 ribu transaksi dari 120 kartu (Sparkov, data kartu SINTETIS, Apache-2.0).

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/tabular_extra/fraud_transaksi"   # GANTI saat lomba
DATA_CSV = f"{DATA_FOLDER}/transaksi_sparkov.csv"

TIME_COL = "waktu_transaksi"
CARD_COL = "cc_num"
AMOUNT_COL = "amt"
TARGET = "is_fraud"
TRAIN_END = "2019-11-01"        # sebelum tanggal ini = train
VALIDATION_END = "2020-01-01"   # sampai sebelum tanggal ini = validasi, sisanya = test
WINDOWS = ["1h", "24h", "7d"]   # jendela waktu feature kecepatan
EARTH_RADIUS_KM = 6371
SEED = 42

## Baca data
Data diurutkan per kartu lalu per waktu, supaya "transaksi sebelumnya" benar.

In [3]:
data = pd.read_csv(DATA_CSV, parse_dates=[TIME_COL])
data = data.sort_values([CARD_COL, TIME_COL])
data = data.reset_index(drop=True)
print(data.shape, "| tingkat fraud:", round(data[TARGET].mean(), 4))
print(data[TIME_COL].min(), "->", data[TIME_COL].max())
print("kartu:", data[CARD_COL].nunique())
data.head()

(132989, 14) | tingkat fraud: 0.0054
2019-01-01 00:00:00 -> 2020-03-10 16:05:00
kartu: 120


,trans_num,waktu_transaksi,cc_num,merchant,category,amt,gender,state,city_pop,lat,long,merch_lat,merch_long,is_fraud
0,6d76dfcd39a8262fa7b8e723e0c6cf7f,2019-01-01 03:12:00,501803000000,"fraud_Rohan, White and Aufderhar",misc_net,8.68,M,CA,308,41.5403,-122.9366,41.013729,-122.403496,0
1,7cb4b8a984ecde859d27017eb2fc2164,2019-01-02 07:23:00,501803000000,fraud_Cummerata-Jones,gas_transport,56.15,M,CA,308,41.5403,-122.9366,42.023583,-123.221480,0
2,e48eb2aa9636cd67e0e50ea92615799b,2019-01-02 07:24:00,501803000000,"fraud_Mosciski, Gislason and Mertz",grocery_pos,166.60,M,CA,308,41.5403,-122.9366,40.605630,-122.103419,0
3,f78975b08ed96c124a92677330b13f83,2019-01-02 16:04:00,501803000000,"fraud_Lowe, Dietrich and Erdman",kids_pets,166.11,M,CA,308,41.5403,-122.9366,40.788462,-122.178549,0
4,e6360079092291d1d26fc7b27bfdab99,2019-01-03 22:31:00,501803000000,"fraud_Nienow, Barrows and Romaguera",personal_care,8.87,M,CA,308,41.5403,-122.9366,42.049686,-122.097644,0


## Feature kecepatan per kartu (hanya masa lalu)
Tiap kartu diproses sendiri. `closed="left"` = jendela TIDAK menyertakan transaksi saat ini.
`shift(1)` = rata-rata hanya dari transaksi sebelumnya.

In [4]:
for window in WINDOWS:
    data["jumlah_" + window] = 0.0
    data["total_" + window] = 0.0
data["jam_sejak_terakhir"] = np.nan
data["rata2_kartu_sebelumnya"] = np.nan

for card, group in data.groupby(CARD_COL):
    for window in WINDOWS:
        rolling = group.rolling(window, on=TIME_COL, closed="left")[AMOUNT_COL]
        data.loc[group.index, "jumlah_" + window] = rolling.count().to_numpy()
        window_total = rolling.sum().fillna(0)
        data.loc[group.index, "total_" + window] = window_total.to_numpy()
    gap = group[TIME_COL].diff()
    data.loc[group.index, "jam_sejak_terakhir"] = gap.dt.total_seconds().to_numpy() / 3600
    past_mean = group[AMOUNT_COL].expanding().mean()
    data.loc[group.index, "rata2_kartu_sebelumnya"] = past_mean.shift(1).to_numpy()

print(data[["jumlah_1h", "jumlah_24h", "total_24h", "jam_sejak_terakhir"]].describe().round(2))

       jumlah_1h  jumlah_24h  total_24h  jam_sejak_terakhir
count   21742.00   121500.00  132989.00           132869.00
mean        1.15        4.30     284.14                9.09
std         0.42        3.03     438.75               13.48
min         1.00        1.00       0.00                0.00
25%         1.00        2.00      66.87                1.67
50%         1.00        4.00     176.30                4.57
75%         1.00        6.00     359.07               11.13
max         6.00       25.00   25352.83              334.92


## Feature lain: rasio nominal, jarak lokasi, waktu, kategori
Jarak rumah pemegang kartu ke merchant (km) dihitung dengan rumus haversine.

In [5]:
data["rasio_nominal"] = data[AMOUNT_COL] / data["rata2_kartu_sebelumnya"]   # kosong di awal
data["jam_hari"] = data[TIME_COL].dt.hour
data["hari_minggu"] = data[TIME_COL].dt.dayofweek

home_lat = np.radians(data["lat"])
shop_lat = np.radians(data["merch_lat"])
lon_gap = np.radians(data["merch_long"] - data["long"])
part = np.sin((shop_lat - home_lat) / 2) ** 2
part = part + np.cos(home_lat) * np.cos(shop_lat) * np.sin(lon_gap / 2) ** 2
data["jarak_km"] = EARTH_RADIUS_KM * 2 * np.arcsin(np.sqrt(part))

data["kategori"] = data["category"].astype("category").cat.codes
data["jenis_kelamin"] = (data["gender"] == "F").astype(int)

VELOCITY_FEATURES = ["jumlah_1h", "total_1h", "jumlah_24h", "total_24h", "jumlah_7d",
                     "total_7d", "jam_sejak_terakhir", "rata2_kartu_sebelumnya", "rasio_nominal"]
BASE_FEATURES = [AMOUNT_COL, "jam_hari", "hari_minggu", "jarak_km", "kategori", "jenis_kelamin",
                 "city_pop"]
print(data[["rasio_nominal", "jarak_km"]].describe().round(2))

       rasio_nominal   jarak_km
count      132869.00  132989.00
mean            1.03      76.42
std             3.15      29.11
min             0.00       0.37
25%             0.16      55.87
50%             0.66      78.44
75%             1.22      98.75
max           676.39     149.43


## Split berdasarkan waktu
Train = masa lalu, validasi = berikutnya, test = paling baru. Split acak akan bocor:
transaksi kartu yang sama di hari yang sama muncul di train dan validasi.

In [6]:
train = data[data[TIME_COL] < TRAIN_END]
validation = data[(data[TIME_COL] >= TRAIN_END) & (data[TIME_COL] < VALIDATION_END)]
test = data[data[TIME_COL] >= VALIDATION_END]
print("train   :", train.shape, "| fraud:", train[TARGET].sum())
print("validasi:", validation.shape, "| fraud:", validation[TARGET].sum())
print("test    :", test.shape, "| fraud:", test[TARGET].sum())

train   : (90296, 28) | fraud: 507
validasi: (27073, 28) | fraud: 81
test    : (15620, 28) | fraud: 127


## Model
`scale_pos_weight` menaikkan bobot kelas fraud yang jarang.
Baris `# features = ...` = tanpa feature kecepatan, untuk melihat seberapa besar sumbangannya.

In [7]:
features = BASE_FEATURES + VELOCITY_FEATURES
# features = BASE_FEATURES

normal_count = (train[TARGET] == 0).sum()
fraud_count = (train[TARGET] == 1).sum()
model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                       scale_pos_weight=normal_count / fraud_count / 5, random_state=SEED,
                       n_jobs=4, verbose=-1)
model.fit(train[features], train[TARGET])

val_proba = model.predict_proba(validation[features])[:, 1]
print("PR-AUC validasi:", round(average_precision_score(validation[TARGET], val_proba), 4))

PR-AUC validasi: 0.965


## Threshold keputusan
PR-AUC tidak butuh threshold, tetapi label akhir butuh. Pilih threshold dengan F1 fraud
terbaik di validasi.

In [8]:
best_f1 = 0
threshold = 0.5
for candidate in np.arange(0.05, 0.96, 0.01):
    candidate_f1 = f1_score(validation[TARGET], val_proba > candidate)
    if candidate_f1 > best_f1:
        best_f1 = candidate_f1
        threshold = candidate

val_guess = val_proba > threshold
print("threshold terbaik:", round(threshold, 2), "| F1 fraud validasi:", round(best_f1, 3))
print("precision:", round(precision_score(validation[TARGET], val_guess), 3),
      "| recall:", round(recall_score(validation[TARGET], val_guess), 3))

importance = pd.Series(model.feature_importances_, index=features)
print(importance.sort_values(ascending=False).head(8))

threshold terbaik: 0.83 | F1 fraud validasi: 0.943
precision: 0.962 | recall: 0.926
amt                       1109
kategori                  1067
rata2_kartu_sebelumnya     911
city_pop                   821
total_24h                  741
total_7d                   657
jam_hari                   626
jumlah_7d                  620
dtype: int32


## Test (data paling baru) dan submission

In [9]:
test_proba = model.predict_proba(test[features])[:, 1]
predictions = (test_proba > threshold).astype(int)
print("PR-AUC test:", round(average_precision_score(test[TARGET], test_proba), 4))
print("F1 fraud test:", round(f1_score(test[TARGET], predictions), 3),
      "| precision:", round(precision_score(test[TARGET], predictions), 3),
      "| recall:", round(recall_score(test[TARGET], predictions), 3))

submission = pd.DataFrame()
submission["trans_num"] = test["trans_num"].to_numpy()
submission["proba_fraud"] = test_proba
submission["is_fraud"] = predictions
submission.to_csv("submission.csv", index=False)
submission.head()

PR-AUC test: 0.9229
F1 fraud test: 0.843 | precision: 0.942 | recall: 0.764


,trans_num,proba_fraud,is_fraud
0,a97d3f8537a4566ef60feac1db04c930,1.793718e-04,0
1,2c52d36725920682abfd55fd2d3465b0,1.447406e-05,0
2,441d1e5d93aada9b9b72e7b6ddc315c6,4.263737e-02,0
3,de0bf7e67465bab9010e6cbf197e3136,8.474655e-07,0
4,410227590fe44784a208e22adfc52382,6.871835e-07,0
